In [1]:
import os
# Limit each parallel process to one thread per library
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['TORCH_NUM_THREADS'] = '1'  # For PyTorch

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, average_precision_score, precision_recall_curve
from joblib import Parallel, delayed
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import warnings
from skrebate import ReliefF
from sklearn.metrics import confusion_matrix

warnings.filterwarnings("ignore")  # Optional: Suppress warnings for cleaner output

# ----------------------------
# Data Preparation
# ----------------------------

# Load the dataset
data = pd.read_excel(r'C:\Users\Inspiron\OneDrive - Loughborough University\Desktop\PhD\articles\prospective study\results\dataset\class 1\class1_dataset.xlsx')

# Define feature groups as per user
feature_groups = {
    'Genotype': [
        'rs11225395', 'rs1144393', 'rs650108', 'rs591058', 'rs2252070', 'rs4986938', 'rs1800012', 'rs4789932', 'rs9340799', 'rs970547', 
        'rs1800795', 'rs13946', 'rs12722', 'class1_SNP_risk_score', 'rs7528684', 'rs4919510', 'rs1937810', 'rs6481512', 'rs1249269', 
        'rs12574452', 'rs12429486', 'rs4454832', 'rs2761884', 'rs62051384', 'rs4362400', 'rs2586488', 'rs2277698', 'rs1045485', 
        'rs143383', 'rs17576', 'rs2305948', 'rs1011814', 'rs11154027', 'rs2234693', 'rs1643821', 'rs2010963', 'rs10263021', 'rs149047058', 
        'rs420257', 'rs42517', 'rs42522', 'rs42531', 'rs413826', 'rs2104772', 'rs1330363', 'class12_SNP_risk_score', 'rs3753841', 
        'rs57104447', 'rs1887632', 'rs4654760', 'rs1137101', 'rs2306033', 'rs2277268', 'rs4988321', 'rs11232681', 'rs1718119', 'rs3751143', 
        'rs1544410', 'rs2228570', 'rs4328262', 'rs1021188', 'rs74544784', 'rs78391032', 'rs77569527', 'rs117544024', 'rs912336', 
        'rs3218791', 'rs911263', 'rs2525504', 'rs17756404', 'rs4903399', 'rs10132091', 'rs17583842', 'rs1676303', 'rs11629171', 
        'rs2281518', 'rs2285053', 'rs71404070', 'rs710079', 'rs2858056', 'rs820218', 'rs3018362', 'rs1800470', 'rs1800469', 'rs25487', 
        'rs25489', 'rs2289360', 'rs183364169', 'rs11177', 'rs6617', 'rs3219008', 'rs13107325', 'rs60713544', 'rs145648292', 'rs4244032', 
        'rs12656106', 'rs3045', 'rs187483', 'rs4701616', 'rs144414988', 'rs1800629', 'rs10484958', 'rs4730153', 'rs1800797', 'rs1554606', 
        'rs2237352', 'rs4725069', 'rs12154667', 'rs1548456', 'rs3216902', 'rs35360670', 'rs13317', 'rs1800972', 'rs7035322', 'rs7021589', 
        'rs72758637', 'rs10759753', 'rs3789870', 'rs1138545', 'rs3196378', 'rs1134170', 'rs10992075', 'rs1590', 'rs144371252', 
        'rs761804508', 'class123_SNP_risk_score', 'sex'
    ],
    'History': [
        'Age', 'lower_limb_days_total', 'average_run_hours', 'average_interval_training_frequency', 'EDEQ_total', 'tracking_period_injury',
        'past_stress_injury', 'LEAF-Q', 'Athlete_Score', 'average_run_frequency', 'past_month_injury'
    ],
    'Phenotype': [
        'hip_abduction_peak_torque', 'total_ad_ab_ratio', 'knee_extension_peak_torque', 'knee_flexion_peak_torque', 'navicular_drop', 
        'navicular_drop_asymmetry', 'Q_angle', 'Q_angle_asymmetry', 'VALR_12', 'Impact_peak_12', 'Duty_factor_12', 'BMI', 'BMD_spine',
        'hip_abduction_peak_torque_asymmetry', 'hip_adduction_peak_torque', 'hip_adduction_peak_torque_asymmetry', 
        'knee_extension_peak_torque_asymmetry', 'knee_flexion_peak_torque_asymmetry', 'total_fl_ex_ratio', 'leg_lean_mass', 
        'hip_abduction_peak_angle', 'hip_abduction_peak_angle_asymmetry', 'hip_adduction_peak_angle', 'hip_adduction_peak_angle_asymmetry', 
        'ad_ab_ratio_asymmetry', 'knee_extension_peak_angle', 'knee_extension_peak_angle_asymmetry', 'knee_flexion_peak_angle', 
        'knee_flexion_peak_angle_asymmetry', 'fl_ex_ratio_asymmetry', 'VILR_10', 'VALR_10', 'VILR_asymmetry_10', 'VALR_asymmetry_10', 
        'Impact_peak_10', 'Impact_peak_asymmetry_10', 'Flight_time_10', 'Contact_time_10', 'Duty_factor_10', 'Step_frequency_10', 
        'Cadence_asymmetry_10', 'Duty_factor_asymmetry_10', 'VILR_12', 'VILR_asymmetry_12', 'VALR_asymmetry_12', 
        'Impact_peak_asymmetry_12', 'Flight_time_12', 'Contact_time_12', 'Step_frequency_12', 'Cadence_asymmetry_12', 
        'Duty_factor_asymmetry_12', 'Alt_strike', 'height', 'Mass', 'thigh_lean_mass', 'thigh_ffmi', 'lower_leg_lean_mass', 
        'lower_leg_ffmi', 'leg_ffmi', 'total_lean_mass', 'total_ffmi', 'calf_size', 'BMD_hip', 'BMD_body',
    ],
    'Behaviour': [
        'fat_intake_avg', 'past_month_distance', 'past_month_ratio', 'SC_past_season', 'non_running_past_season', 'fat_intake_BW', 
        'fat_percentage_avg', 'average_energy_availability', 'protein_intake_BW', 'omega3_intake_BW', 'vitaminD_intake_BW', 
        'vitaminC_intake_BW', 'vitaminE_intake_BW', 'calcium_intake_BW', 'copper_intake_BW', 'iron_intake_BW', 'glycine_intake_BW', 
        'arginine_intake_BW', 'past_month_min', 'past_week_ratio', 'past_month_volume_low', 'past_week_ratio_low', 'past_month_ratio_low', 
        'past_month_volume_moderate', 'past_week_ratio_moderate', 'past_month_ratio_moderate', 'past_month_volume_high', 
        'past_week_ratio_high', 'past_month_ratio_high', 'past_month_volume_very_high', 'past_week_ratio_very_high', 
        'past_month_ratio_very_high', 'past_month_calculated_volume', 'past_week_ratio_calculated_volume', 
        'past_month_ratio_calculated_volume', 'resistance_training_past_month', 'resistance_training_past_season', 
        'bodyweight_exercises_past_month', 'bodyweight_exercises_past_season', 'core_stability_past_month', 'core_stability_past_season', 
        'balance_training_past_month', 'balance_training_past_season', 'plyometrics_past_month', 'plyometrics_past_season', 
        'drills_past_month', 'drills_past_season', 'circuit_training_past_month', 'circuit_training_past_season', 'barefoot_past_month', 
        'barefoot_past_season', 'stretching_past_month', 'stretching_past_season', 'SC_past_month', 'non_running_past_month'
    ]
}

# Define predictors and outcome
X = data.drop(columns=['RRI'])  # Predictors
y = data['RRI']  # Outcome

# Ensure that the feature groups exist in the dataset
for group in feature_groups:
    feature_groups[group] = [feature for feature in feature_groups[group] if feature in X.columns]

# ----------------------------
# Per-Group ReliefF Feature Ranking
# ----------------------------

# Initialize a dictionary to hold ranked features per group
group_ranked_features = {}

# Initialize a dictionary to hold ranked feature scores per group
group_ranked_scores = {}

# Apply ReliefF separately to each feature group
for group, features in feature_groups.items():
    if len(features) == 0:
        print(f"Warning: No features found in group '{group}'. Skipping.")
        continue
    X_group = X[features].values
    y_group = y.values
    relief = ReliefF(n_neighbors=100, n_jobs=-1)
    relief.fit(X_group, y_group)
    feature_scores = pd.Series(relief.feature_importances_, index=features)
    ranked_features = feature_scores.sort_values(ascending=False)
    group_ranked_features[group] = ranked_features.index.tolist()
    group_ranked_scores[group] = ranked_features
    print(f"Group '{group}' Ranked Features ({len(ranked_features)}):")
    print(ranked_features)
    print("\n")

# ----------------------------
# Select Features Based on Ranked Groups
# ----------------------------

# Best hyperparameter subset
best_params = {
    'n_genotype': 13,
    'n_history': 3,
    'n_phenotype': 11,
    'n_behaviour': 3,
    'learning_rate': 0.0013776008234454911,
    'epochs': 2382,
    'batch_size': 32
}

device = 'cpu'

# Select features based on the number of features per group
selected_genotype_features = group_ranked_features['Genotype'][:best_params['n_genotype']]
selected_history_features = group_ranked_features['History'][:best_params['n_history']]
selected_phenotype_features = group_ranked_features['Phenotype'][:best_params['n_phenotype']]
selected_behaviour_features = group_ranked_features['Behaviour'][:best_params['n_behaviour']]

# Combine selected features
current_selected_features = selected_genotype_features + selected_history_features + \
                             selected_phenotype_features + selected_behaviour_features

print("Selected Features based on Best Hyperparameters:")
print(current_selected_features)
print("\n")

# Prepare data based on current_selected_features
X_current = X[current_selected_features].copy()

# Split feature groups
X_genotype_current = X_current[selected_genotype_features].values.astype(np.float32)
X_history_current = X_current[selected_history_features].values.astype(np.float32)
X_phenotype_current = X_current[selected_phenotype_features].values.astype(np.float32)
X_behaviour_current = X_current[selected_behaviour_features].values.astype(np.float32)

# Convert target to numpy array
y_current = y.values.astype(np.float32)

# ----------------------------
# Dataset and DataLoader
# ----------------------------

class CustomDataset(Dataset):
    def __init__(self, genotype, history, phenotype, behaviour, labels):
        self.genotype = genotype
        self.history = history
        self.phenotype = phenotype
        self.behaviour = behaviour
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            self.genotype[idx],
            self.history[idx],
            self.phenotype[idx],
            self.behaviour[idx],
            self.labels[idx]
        )

# ----------------------------
# FeatureAttention and MaskedLinear Layers
# ----------------------------

class FeatureAttention(nn.Module):
    def __init__(self, feature_dim):
        super(FeatureAttention, self).__init__()
        self.attention = nn.Sequential(
            nn.Linear(feature_dim, max(1, feature_dim // 2)),
            nn.ReLU(),
            nn.Linear(max(1, feature_dim // 2), feature_dim),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        weights = self.attention(x)
        return x * weights

class MaskedLinear(nn.Module):
    def __init__(self, in_features, out_features, bias=True):
        super(MaskedLinear, self).__init__()
        # Initialize the linear layer
        self.linear = nn.Linear(in_features, out_features, bias)
        # Initialize mask parameters with the same shape as weights
        self.mask_param = nn.Parameter(torch.ones_like(self.linear.weight))
        if bias:
            self.bias = self.linear.bias
        else:
            self.register_parameter('bias', None)
    
    def forward(self, x):
        # Apply sigmoid to mask parameters to get gating probabilities
        mask = torch.sigmoid(self.mask_param)
        # Apply the mask to the weights
        masked_weight = self.linear.weight * mask
        return nn.functional.linear(x, masked_weight, self.bias)
    
    def get_binary_mask(self, threshold=0.5):
        """
        Returns a binary mask based on the gating parameters and a specified threshold.
        """
        with torch.no_grad():
            mask = torch.sigmoid(self.mask_param)
            binary_mask = (mask > threshold).float()
        return binary_mask

# ----------------------------
# Modified Model Definition with Optional Attention and Mask
# ----------------------------

class CustomMLPWithOptionalComponents(nn.Module):
    def __init__(self, genotype_size, history_size, phenotype_size, behaviour_size,
                 use_attention=True, use_mask=False):
        super(CustomMLPWithOptionalComponents, self).__init__()
        
        self.use_attention = use_attention
        self.use_mask = use_mask

        # Attention for Genotype Features
        if self.use_attention:
            self.attention_genotype = FeatureAttention(genotype_size)
        
        # Define the main layers with MaskedLinear or Linear based on use_mask
        LinearLayer = MaskedLinear if self.use_mask else nn.Linear

        # Genotype to History
        self.genotype_to_history = LinearLayer(genotype_size, history_size, bias=False)
        self.bn_genotype_to_history = nn.BatchNorm1d(history_size)
        self.genotype_to_history_bias = nn.Parameter(torch.zeros(history_size))
        if self.use_attention:
            self.attention_history = FeatureAttention(history_size + 1)  # +1 for extra input

        # History to Phenotype
        self.history_to_pheno = LinearLayer(history_size + 1, phenotype_size, bias=False)
        self.bn_history_to_pheno = nn.BatchNorm1d(phenotype_size)
        self.history_to_pheno_bias = nn.Parameter(torch.zeros(phenotype_size))
        if self.use_attention:
            self.attention_pheno = FeatureAttention(phenotype_size + 1)
        
        # Phenotype to Behaviour
        self.pheno_to_behaviour = LinearLayer(phenotype_size + 1, behaviour_size, bias=False)
        self.bn_pheno_to_behaviour = nn.BatchNorm1d(behaviour_size)
        self.pheno_to_behaviour_bias = nn.Parameter(torch.zeros(behaviour_size))
        if self.use_attention:
            self.attention_behaviour = FeatureAttention(behaviour_size + 1)
        
        # Behaviour to Output
        self.behaviour_to_output = LinearLayer(behaviour_size + 1, 1)
        
        # Extra regular node layers (no batch norm needed)
        self.genotype_to_history_extra = LinearLayer(genotype_size, 1, bias=True)
        self.history_to_pheno_extra = LinearLayer(history_size + 1, 1, bias=True)
        self.pheno_to_behaviour_extra = LinearLayer(phenotype_size + 1, 1, bias=True)
        
        # Activation function
        self.relu = torch.nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        
        # Initialize weights using Xavier/Glorot initialization
        for layer in [
            self.genotype_to_history, 
            self.history_to_pheno, 
            self.pheno_to_behaviour, 
            self.behaviour_to_output,
            self.genotype_to_history_extra,
            self.history_to_pheno_extra,
            self.pheno_to_behaviour_extra
        ]:
            if self.use_mask and isinstance(layer, MaskedLinear):
                nn.init.xavier_uniform_(layer.linear.weight)
                if layer.linear.bias is not None:
                    nn.init.zeros_(layer.linear.bias)
            elif isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                if layer.bias is not None:
                    nn.init.zeros_(layer.bias)
    
    def forward(self, genotype, history, phenotype, behaviour):
        # Genotype to History with optional attention
        if self.use_attention:
            genotype_att = self.attention_genotype(genotype)
        else:
            genotype_att = genotype
        main_history_input = self.genotype_to_history(genotype_att) * history
        main_history_input = self.bn_genotype_to_history(main_history_input)
        main_history_input = main_history_input + self.genotype_to_history_bias
        
        # Extra input (no batch norm)
        extra_history_input = self.genotype_to_history_extra(genotype_att)
        combined_history_input = torch.cat([main_history_input, extra_history_input], dim=1)
        history_output = self.relu(combined_history_input)
        
        # History to Phenotype with optional attention
        if self.use_attention:
            history_att = self.attention_history(history_output)
        else:
            history_att = history_output
        main_pheno_input = self.history_to_pheno(history_att) * phenotype
        main_pheno_input = self.bn_history_to_pheno(main_pheno_input)
        main_pheno_input = main_pheno_input + self.history_to_pheno_bias
        
        # Extra input for phenotype (no batch norm)
        extra_pheno_input = self.history_to_pheno_extra(history_att)
        combined_pheno_input = torch.cat([main_pheno_input, extra_pheno_input], dim=1)
        pheno_output = self.relu(combined_pheno_input) 
        
        # Phenotype to Behaviour with optional attention
        if self.use_attention:
            pheno_att = self.attention_pheno(pheno_output)
        else:
            pheno_att = pheno_output
        main_behaviour_input = self.pheno_to_behaviour(pheno_att) * behaviour
        main_behaviour_input = self.bn_pheno_to_behaviour(main_behaviour_input)
        main_behaviour_input = main_behaviour_input + self.pheno_to_behaviour_bias

        # Extra input for behaviour (no batch norm)
        extra_behaviour_input = self.pheno_to_behaviour_extra(pheno_att)
        combined_behaviour_input = torch.cat([main_behaviour_input, extra_behaviour_input], dim=1)
        behaviour_output = self.relu(combined_behaviour_input) 
        
        # Behaviour to Output with optional attention
        if self.use_attention:
            behaviour_att = self.attention_behaviour(behaviour_output)
        else:
            behaviour_att = behaviour_output
        final_input = self.behaviour_to_output(behaviour_att)
        final_output = self.sigmoid(final_input)
        
        return final_output

# ----------------------------
# Cross-Validation Setup
# ----------------------------
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Storage for metrics and predictions
all_actuals = []
all_probs = []
fold_metrics = {
    'auc': [],
    'auprc': [],
    'f1': [],
    'accuracy': [],
    'precision': []
}

# ----------------------------
# Cross-Validation Loop
# ----------------------------
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    print(f"\nTraining Fold {fold+1}/10")
    
    # Split data
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    # Select features using ReliefF for this fold
    current_selected_features = []
    for group in ['Genotype', 'History', 'Phenotype', 'Behaviour']:
        if group in group_ranked_features:
            n_features = best_params[f'n_{group.lower()}']
            current_selected_features.extend(group_ranked_features[group][:n_features])
    
    # Prepare fold data
    X_train_fold = X_train[current_selected_features].values.astype(np.float32)
    X_test_fold = X_test[current_selected_features].values.astype(np.float32)
    y_train_fold = y_train.values.astype(np.float32)
    y_test_fold = y_test.values.astype(np.float32)
    
    # Split into feature groups
    X_train_gen = X_train_fold[:, :best_params['n_genotype']]
    X_train_hist = X_train_fold[:, best_params['n_genotype']:best_params['n_genotype']+best_params['n_history']]
    X_train_pheno = X_train_fold[:, best_params['n_genotype']+best_params['n_history']:best_params['n_genotype']+best_params['n_history']+best_params['n_phenotype']]
    X_train_behav = X_train_fold[:, best_params['n_genotype']+best_params['n_history']+best_params['n_phenotype']:]
    
    X_test_gen = X_test_fold[:, :best_params['n_genotype']]
    X_test_hist = X_test_fold[:, best_params['n_genotype']:best_params['n_genotype']+best_params['n_history']]
    X_test_pheno = X_test_fold[:, best_params['n_genotype']+best_params['n_history']:best_params['n_genotype']+best_params['n_history']+best_params['n_phenotype']]
    X_test_behav = X_test_fold[:, best_params['n_genotype']+best_params['n_history']+best_params['n_phenotype']:]
    
    # Create DataLoaders
    train_dataset = CustomDataset(X_train_gen, X_train_hist, X_train_pheno, X_train_behav, y_train_fold)
    test_dataset = CustomDataset(X_test_gen, X_test_hist, X_test_pheno, X_test_behav, y_test_fold)
    
    train_loader = DataLoader(train_dataset, batch_size=best_params['batch_size'], shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=best_params['batch_size'], shuffle=False)
    
    # Initialize model
    model = CustomMLPWithOptionalComponents(
        genotype_size=best_params['n_genotype'],
        history_size=best_params['n_history'],
        phenotype_size=best_params['n_phenotype'],
        behaviour_size=best_params['n_behaviour'],
        use_attention=True,
        use_mask=False
    ).to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=best_params['learning_rate'])
    criterion = nn.BCELoss()
    
    # Training
    model.train()
    for epoch in tqdm(range(best_params['epochs']), desc=f"Fold {fold+1} Epochs"):
        for batch in train_loader:
            genotype, history, phenotype, behaviour, labels = batch
            genotype = genotype.to(device)
            history = history.to(device)
            phenotype = phenotype.to(device)
            behaviour = behaviour.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(genotype, history, phenotype, behaviour)
            loss = criterion(outputs.squeeze(), labels)
            loss.backward()
            optimizer.step()
    
    # Evaluation
    model.eval()
    fold_probs = []
    fold_actuals = []
    with torch.no_grad():
        for batch in test_loader:
            genotype, history, phenotype, behaviour, labels = batch
            genotype = genotype.to(device)
            history = history.to(device)
            phenotype = phenotype.to(device)
            behaviour = behaviour.to(device)
            
            outputs = model(genotype, history, phenotype, behaviour)
            fold_probs.extend(outputs.cpu().numpy().flatten().tolist())
            fold_actuals.extend(labels.cpu().numpy().flatten().tolist())
    
    # Store results
    all_actuals.extend(fold_actuals)
    all_probs.extend(fold_probs)
    
    # Calculate fold metrics
    fold_auc = roc_auc_score(fold_actuals, fold_probs)
    fold_auprc = average_precision_score(fold_actuals, fold_probs)
    fold_metrics['auc'].append(fold_auc)
    fold_metrics['auprc'].append(fold_auprc)

# ----------------------------
# Find Optimal Threshold
# ----------------------------
precision, recall, thresholds = precision_recall_curve(all_actuals, all_probs)
f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-9)
best_threshold = thresholds[np.argmax(f1_scores)]

# ----------------------------
# Recalculate Metrics with Optimal Threshold (including sensitivity/specificity)
# ----------------------------
print("\nRecalculating metrics with optimal threshold...")
for fold in range(10):
    start = len(all_actuals)//10 * fold
    end = len(all_actuals)//10 * (fold+1)
    fold_actuals = all_actuals[start:end]
    fold_probs = all_probs[start:end]
    
    y_pred = (np.array(fold_probs) >= best_threshold).astype(int)
    
    # Calculate standard metrics
    fold_metrics['f1'].append(f1_score(fold_actuals, y_pred))
    fold_metrics['accuracy'].append(accuracy_score(fold_actuals, y_pred))
    fold_metrics['precision'].append(precision_score(fold_actuals, y_pred, zero_division=0))
    
    # Calculate sensitivity (recall) and specificity
    tn, fp, fn, tp = confusion_matrix(fold_actuals, y_pred).ravel()
    sensitivity = tp / (tp + fn + 1e-9)  # Avoid division by zero
    specificity = tn / (tn + fp + 1e-9)  # Avoid division by zero
    
    # Store in fold_metrics (adding new keys if they don't exist)
    if 'sensitivity' not in fold_metrics:
        fold_metrics['sensitivity'] = []
        fold_metrics['specificity'] = []
    fold_metrics['sensitivity'].append(sensitivity)
    fold_metrics['specificity'].append(specificity)

# ----------------------------
# Calculate Statistics (updated with sensitivity/specificity)
# ----------------------------
def format_metric(mean, std):
    return f"{mean:.4f} ± {std:.4f}"

print("\nFinal Metrics:")
print(f"Optimal Threshold: {best_threshold:.4f}")
print(f"AUC: {format_metric(np.mean(fold_metrics['auc']), np.std(fold_metrics['auc']))}")
print(f"AUPRC: {format_metric(np.mean(fold_metrics['auprc']), np.std(fold_metrics['auprc']))}")
print(f"F1 Score: {format_metric(np.mean(fold_metrics['f1']), np.std(fold_metrics['f1']))}")
print(f"Accuracy: {format_metric(np.mean(fold_metrics['accuracy']), np.std(fold_metrics['accuracy']))}")
print(f"Precision: {format_metric(np.mean(fold_metrics['precision']), np.std(fold_metrics['precision']))}")
print(f"Sensitivity (Recall): {format_metric(np.mean(fold_metrics['sensitivity']), np.std(fold_metrics['sensitivity']))}")
print(f"Specificity: {format_metric(np.mean(fold_metrics['specificity']), np.std(fold_metrics['specificity']))}")

Group 'Genotype' Ranked Features (15):
rs2252070                0.242665
rs4986938                0.242034
rs12722                  0.240362
rs1144393                0.234960
rs591058                 0.231173
rs4789932                0.227995
rs11225395               0.220529
rs13946                  0.219393
class1_SNP_risk_score    0.210262
rs9340799                0.202750
rs970547                 0.192234
rs1800012                0.159552
sex                      0.150060
rs1800795                0.124130
rs650108                 0.057680
dtype: float64


Group 'History' Ranked Features (6):
average_run_hours                      0.070007
Age                                    0.063228
average_interval_training_frequency    0.056251
EDEQ_total                             0.052285
tracking_period_injury                 0.032576
lower_limb_days_total                  0.027903
dtype: float64


Group 'Phenotype' Ranked Features (13):
BMD_spine                     0.091148
Q_angle_asymm

Fold 1 Epochs: 100%|█████████████████████████████████████████████████████████████| 2382/2382 [1:55:26<00:00,  2.91s/it]



Training Fold 2/10


Fold 2 Epochs: 100%|█████████████████████████████████████████████████████████████| 2382/2382 [1:33:07<00:00,  2.35s/it]



Training Fold 3/10


...


Training Fold 9/10


Fold 9 Epochs: 100%|███████████████████████████████████████████████████████████████| 2382/2382 [53:24<00:00,  1.35s/it]



Training Fold 10/10


Fold 10 Epochs: 100%|██████████████████████████████████████████████████████████████| 2382/2382 [53:16<00:00,  1.34s/it]


Recalculating metrics with optimal threshold...

Final Metrics:
Optimal Threshold: 0.1833
AUC: 0.7197 ± 0.0379
AUPRC: 0.2508 ± 0.0570
F1 Score: 0.3102 ± 0.0605
Accuracy: 0.8487 ± 0.0121
Precision: 0.2653 ± 0.0476
Sensitivity (Recall): 0.3756 ± 0.0836
Specificity: 0.8962 ± 0.0110
